# 02 — Silver: Café Enchancia
**User story:** As an analyst, I want typed, validated orders with consistent reporting dates and peso amounts so I can compare stores correctly.

Run your updated Bronze notebook first. This notebook reads `ws_tip_demo.<target_schema>.ec_bronze_<store>` and writes `ec_silver_<store>` in the same schema. Default: KR_SEOUL. Source Volume paths are handled by Bronze and are not needed here.

Keep every order, including CANCELLED. Keep source currency, local prices, local discounts and timestamp_local. Add UTC transaction time, New York reporting date, and clearly labelled PHP reporting measures. There is no inventory, deduplication or dirty-data exercise.

**Acceptance:** Bronze and Silver counts match; identifiers are unique; required values and types are valid; cancellations contribute zero sales; local amounts remain traceable; peso values follow the stated rate. Fail before writing when a check fails.

Use assigned Spark compute supporting `try_to_timestamp` (Spark 3.5+ / compatible Databricks serverless). You need permission to read Bronze and create/overwrite your Silver table. Reruns replace only the chosen store's complete workshop batch. Set target_schema to your assigned group schema for isolation.


## Workshop setup and scope
Set **store** and **target_schema** before running, then run the configuration cell and all following cells. Use the same values across Bronze, Silver and Gold. The instructor default schema is `tip_demo_schema`; students use their assigned `tip_group_01` through `tip_group_10`. Schemas and permissions must be provisioned before class. Source files remain in the shared training Volumes, regardless of the target schema.

Run Bronze, then Silver, then Gold for one assigned store. Gold `ALL` is an instructor consolidation option after all seven stores have been loaded in the same target schema. A one-store run creates three tables; a full seven-store run creates 15 tables: seven Bronze, seven Silver, one Gold.

This is a fixed seven-day archive exercise. No direct POS, email or network-drive connection is made by students. Scheduling the 8 a.m. New York report, arrival/completeness monitoring, alerts, incremental history, corrected-file handling and refunds remain follow-up backlog items. Tax is excluded in the fictional data. All original files for the selected store must remain present for a full-batch rerun.


In [ ]:
dbutils.widgets.text("store", "KR_SEOUL", "Store ID (Korea: KR_SEOUL)")

dbutils.widgets.text("target_schema", "tip_demo_schema", "Target schema (use your assigned group schema)")


## 1. Configure the selected store
Use the same store value as Bronze. `korea` and `seoul` are aliases for KR_SEOUL. Run separately for each store. Set **target_schema** to your assigned group schema, using the same value in every notebook.

Fixed workshop FX rates are PHP per source-currency unit: PHP 1, USD 60, KRW 0.04. These are fictional teaching assumptions, not live market rates. One Americano becomes ₱165 from PHP 165, ₱600 from USD 10, and ₱300 from KRW 7,500, before discounts.


In [ ]:
from pyspark.sql import functions as F
from decimal import Decimal

STORE_CONFIG = {
    "KR_SEOUL": ("KRW", "Asia/Seoul", "0.04"),
    "PH_LUZON": ("PHP", "Asia/Manila", "1.00"),
    "PH_VIS": ("PHP", "Asia/Manila", "1.00"),
    "PH_MIND": ("PHP", "Asia/Manila", "1.00"),
    "US_NY": ("USD", "America/New_York", "60.00"),
    "US_QUEENS": ("USD", "America/New_York", "60.00"),
    "US_NJ": ("USD", "America/New_York", "60.00")
}
value = dbutils.widgets.get("store").strip().upper()
STORE = {"KOREA": "KR_SEOUL", "SEOUL": "KR_SEOUL"}.get(value, value)
if STORE not in STORE_CONFIG:
    raise ValueError(f"Use one of {sorted(STORE_CONFIG)}")
CURRENCY, TIME_ZONE, RATE = STORE_CONFIG[STORE]
CATALOG = "ws_tip_demo"
SCHEMA = dbutils.widgets.get("target_schema").strip()
import re
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", SCHEMA):
    raise ValueError("target_schema must be a valid assigned schema name.")
if not spark.catalog.databaseExists(f"{CATALOG}.{SCHEMA}"):
    raise ValueError(f"Schema {CATALOG}.{SCHEMA} is unavailable. Ask your instructor to provision it.")
BRONZE_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`ec_bronze_{STORE.lower()}`"
SILVER_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`ec_silver_{STORE.lower()}`"
spark.conf.set("spark.sql.session.timeZone", "UTC")
print(f"Read: {BRONZE_TABLE}\nWrite: {SILVER_TABLE}\nRate: {RATE} PHP per {CURRENCY}")


## 2. Read Bronze and check the source contract
No filtering removes source records. Check the store, currency and timezone before applying conversions. This prevents conversion with a mismatched source currency.


In [ ]:
bronze = spark.table(BRONZE_TABLE)
required = ["store_id", "store_name", "country", "time_zone", "currency", "business_date",
            "timestamp_local", "transaction_id", "order_status", "product_name", "category",
            "quantity", "unit_price", "discount_amount", "source_file", "source_path", "ingested_at_utc"]
missing = set(required) - set(bronze.columns)
assert not missing, f"Missing Bronze columns: {missing}"
source_count = bronze.count()
assert source_count > 0, "Bronze is empty."
valid_source = ((F.col("store_id") == STORE) & (F.col("currency") == CURRENCY)
                & (F.col("time_zone") == TIME_ZONE))
for name in required:
    valid_source = valid_source & F.col(name).isNotNull() & (F.trim(F.col(name).cast("string")) != "")
assert bronze.filter(~F.coalesce(valid_source, F.lit(False))).count() == 0, "Check required fields, store, currency and timezone."
assert bronze.select("transaction_id").distinct().count() == source_count, "Duplicate transaction IDs: investigate the source."


## 3. Convert types and derive the reporting date
The original timestamp_local text stays unchanged. A temporary wall-clock value is parsed under the UTC session, interpreted in the store's timezone, then converted to the actual UTC instant. The New York date is derived from that instant.

Keep both local business_date and report_date_ny. An Asian sale shortly after local midnight belongs to the previous New York day. These October samples avoid the repeated local hour during daylight-saving fall-back; production data covering that hour also needs a source UTC offset or event instant.


In [ ]:
typed = (bronze
    .withColumn("business_date", F.expr("try_cast(business_date as date)"))
    .withColumn("quantity", F.expr("try_cast(quantity as int)"))
    .withColumn("unit_price", F.expr("try_cast(unit_price as decimal(18,2))"))
    .withColumn("discount_amount", F.expr("try_cast(discount_amount as decimal(18,2))"))
    .withColumn("_local_wall_clock", F.try_to_timestamp("timestamp_local", F.lit("yyyy-MM-dd HH:mm:ss")))
    .withColumn("timestamp_utc", F.to_utc_timestamp("_local_wall_clock", TIME_ZONE))
    .withColumn("report_date_ny", F.to_date(F.from_utc_timestamp("timestamp_utc", "America/New_York"))))
valid = (F.col("business_date").isNotNull() & F.col("timestamp_utc").isNotNull()
    & (F.to_date("_local_wall_clock") == F.col("business_date"))
    & (F.col("quantity") > 0) & (F.col("unit_price") >= 0)
    & (F.col("discount_amount") >= 0)
    & (F.col("discount_amount") <= F.col("quantity") * F.col("unit_price"))
    & F.col("order_status").isin("COMPLETED", "CANCELLED"))
assert typed.filter(~F.coalesce(valid, F.lit(False))).count() == 0, "Invalid types, dates, quantities, discounts or status."


## 4. Add peso reporting measures
`currency`, `unit_price` and `discount_amount` still describe source-local money.

- quoted_amount_local = quantity × unit_price − discount_amount for any order.
- net_sales_local = quoted amount for COMPLETED; zero for CANCELLED.
- net_sales_php = net_sales_local × fx_rate_to_php, rounded to two decimals **per order**.

The rate and `reporting_currency = PHP` are saved beside each order. Real reporting would require approved, date-effective rates. No refunds or tax calculations are implemented.


In [ ]:
silver = (typed.drop("_local_wall_clock")
    .withColumn("fx_rate_to_php", F.lit(Decimal(RATE)).cast("decimal(10,4)"))
    .withColumn("fx_rate_basis", F.lit("FIXED_WORKSHOP_RATE"))
    .withColumn("reporting_currency", F.lit("PHP"))
    .withColumn("quoted_amount_local",
                (F.col("quantity") * F.col("unit_price") - F.col("discount_amount")).cast("decimal(24,2)"))
    .withColumn("net_sales_local", F.when(F.col("order_status") == "COMPLETED", F.col("quoted_amount_local"))
                .otherwise(F.lit(0)).cast("decimal(24,2)"))
    .withColumn("net_sales_php", F.round(F.col("net_sales_local") * F.col("fx_rate_to_php"), 2).cast("decimal(28,2)")))
assert silver.filter(F.col("net_sales_php").isNull()).count() == 0
assert silver.count() == source_count
(silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE))


## 5. Check the saved Silver table
The final comparison checks every persisted field against the transformation. Inspect local and New York dates side by side. Rerun with unchanged Bronze and verify stable counts and amounts.


In [ ]:
saved = spark.table(SILVER_TABLE)
assert saved.count() == source_count
assert saved.select("transaction_id").distinct().count() == source_count
assert silver.exceptAll(saved.select(*silver.columns)).limit(1).count() == 0
assert saved.select(*silver.columns).exceptAll(silver).limit(1).count() == 0
assert saved.filter("order_status = 'CANCELLED' AND (net_sales_local != 0 OR net_sales_php != 0)").count() == 0
assert saved.filter(F.col("net_sales_php") != F.round(F.col("net_sales_local") * F.col("fx_rate_to_php"), 2)).count() == 0
display(saved.select("store_id", "timestamp_local", "business_date", "timestamp_utc", "report_date_ny",
                     "product_name", "currency", "unit_price", "order_status", "net_sales_local", "net_sales_php")
        .orderBy("timestamp_local").limit(20))
display(saved.groupBy("store_id", "currency", "reporting_currency", "order_status")
        .agg(F.count("*").alias("orders"), F.sum("net_sales_php").alias("net_sales_php")))
print(f"Silver checks passed: {source_count:,} orders saved to {SILVER_TABLE}.")
